In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Define the project root (go up one level from notebooks/)
PROJECT_ROOT = Path.cwd().parent

# 2. Load the data using the absolute path
df = pd.read_parquet(PROJECT_ROOT / "data/processed/listings_clean.parquet")
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")

Rows: 16,558 | Columns: 94


In [4]:
TARGET = "price"

NUMERIC_FEATURES = [
    "accommodates", "bedrooms", "beds",
    "minimum_nights", "amenity_count",
    "distance_to_center_km", "latitude", "longitude",
]

CATEGORICAL_FEATURES = [
    "room_type", "property_type", "neighbourhood_cleansed",
]

print("Numeric:", NUMERIC_FEATURES)
print("Categorical:", CATEGORICAL_FEATURES)
print("\nMissing value counts:")
print(df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].isna().sum())

Numeric: ['accommodates', 'bedrooms', 'beds', 'minimum_nights', 'amenity_count', 'distance_to_center_km', 'latitude', 'longitude']
Categorical: ['room_type', 'property_type', 'neighbourhood_cleansed']

Missing value counts:
accommodates                0
bedrooms                    5
beds                      146
minimum_nights              0
amenity_count               0
distance_to_center_km       0
latitude                    0
longitude                   0
room_type                   0
property_type               0
neighbourhood_cleansed      0
dtype: int64


In [5]:
X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

Train: 13,246 | Test: 3,312


In [6]:
median_price = y_train.median()
baseline_pred = np.full_like(y_test, fill_value=median_price)

baseline_metrics = {
    "model": "median_baseline",
    "mae":  float(mean_absolute_error(y_test, baseline_pred)),
    "rmse": float(np.sqrt(mean_squared_error(y_test, baseline_pred))),
    "r2":   float(r2_score(y_test, baseline_pred)),
    "median_train_price": float(median_price),
}
print(baseline_metrics)

{'model': 'median_baseline', 'mae': 3196.5243146135267, 'rmse': 8481.946677619704, 'r2': -0.039795490150339674, 'median_train_price': 4495.835}


### Baseline — predict the training median

- **Median training price:** 4,495.84 TRY
- **Baseline MAE:** 3,196.52 TRY (the "no model" cost)
- **Baseline RMSE:** 8,481.95 TRY
- **Baseline R²:** -0.04 
- **Any model must beat this on the test set to be useful.**

*(Note: The R² is slightly negative, which is expected. It means that predicting the median on unseen data performs slightly worse than just predicting the mean of those unseen values — essentially confirming that the median is a dumb, static guess with no predictive power).*

In [7]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve() / "backend"))
from app.ml.training import (
    build_linear_pipeline, build_ridge_pipeline,
    build_histgb_pipeline, compare_models,
)

models = {
    "linear": build_linear_pipeline(),
    "ridge": build_ridge_pipeline(),
    "histgb": build_histgb_pipeline(),
}
results = compare_models(models, X_train, y_train, X_test, y_test)

# Add baseline to the table for comparison
results_with_baseline = pd.concat([
    pd.DataFrame([baseline_metrics])[["model", "mae", "rmse", "r2"]],
    results,
]).reset_index(drop=True)

print(results_with_baseline.to_string(index=False))

Fitting linear...
Fitting ridge...
Fitting histgb...
          model         mae        rmse        r2
median_baseline 3196.524315 8481.946678 -0.039795
         histgb 1981.275176 7242.176955  0.241955
          ridge 2496.211121 7779.518949  0.125294
         linear 2506.450030 7786.613611  0.123697
